In [5]:
# Step 1: Import the libraries we need
# Why this step comes first:
# Before we build any neural network, we must import the tools.
# These libraries help us load data, scale features, build the model, and measure results.

import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

In [7]:
# Step 2: Load the dataset
# Why this step comes after Step 1:
# In Step 1 we imported all the tools.
# Now we use those tools to bring the dataset into the notebook.
# Before building a neural network, we must first check whether the data is clean.

ALLData = pd.read_csv("All_features.csv", index_col="Date", parse_dates=["Date"]).dropna()

print("Shape:", ALLData.shape)
print("\nDate Range:", ALLData.index.min(), "to", ALLData.index.max())
print("\nNull Count:")
print(ALLData.isnull().sum())
print("\nDuplicates:", ALLData.index.duplicated().sum())

print("\nFirst 5 rows:")
print(ALLData.head())

Shape: (2342, 17)

Date Range: 2015-01-05 00:00:00 to 2025-12-30 00:00:00

Null Count:
HDFCBANK     0
NIFTY50      0
BANKNIFTY    0
SENSEX       0
INDIA_VIX    0
USDINR       0
SP500        0
NASDAQ100    0
DOWJONES     0
NIKKEI225    0
HANGSENG     0
GOLD         0
BRENT_OIL    0
ICICIBANK    0
SBIN         0
AXISBANK     0
KOTAKBANK    0
dtype: int64

Duplicates: 0

First 5 rows:
              HDFCBANK    NIFTY50    BANKNIFTY      SENSEX   INDIA_VIX  \
Date                                                                     
2015-01-05  500.999146  53.110001  1203.900024  218.399033  302.995667   
2015-01-06  483.090759  51.099998  1219.300049  214.999222  290.143066   
2015-01-07  482.703583  51.150002  1210.599976  215.626694  282.297943   
2015-01-08  485.898041  50.959999  1208.400024  220.155975  289.976135   
2015-01-09  479.121918  50.110001  1216.000000  222.620316  285.302460   

               USDINR       SP500   NASDAQ100      DOWJONES     NIKKEI225  \
Date               

In [9]:
# Step 3: Create the target column
# Why this step comes after Step 2:
# In Step 2 we loaded and checked the dataset.
# Now we must define what the neural network should predict.
# Our goal is to predict whether HDFCBANK will go up or down the next day.

ALLData["HDFC_ret_next"] = ALLData["HDFCBANK"].pct_change().shift(-1)

# This line creates the next-day percentage return of HDFCBANK.
# pct_change() gives daily return.
# shift(-1) moves tomorrow's return to today's row.
# So each row now contains the future return that we want to predict.

ALLData["Target"] = (ALLData["HDFC_ret_next"] > 0).astype(int)

# This line converts the future return into a binary class.
# If next day's return is positive, Target = 1
# Otherwise, Target = 0

ALLData = ALLData.dropna(subset=["HDFC_ret_next"])

# The last row becomes empty after shift(-1) because there is no next day.
# So we remove that row.

print("Shape after target creation:", ALLData.shape)

print("\nTarget value counts:")
print(ALLData["Target"].value_counts())

print("\nTarget percentage:")
print(round(ALLData["Target"].value_counts(normalize=True) * 100, 2))

print("\nFirst 5 rows of target check:")
print(ALLData[["HDFCBANK", "HDFC_ret_next", "Target"]].head())

Shape after target creation: (2341, 19)

Target value counts:
1    1212
0    1129
Name: Target, dtype: int64

Target percentage:
1    51.77
0    48.23
Name: Target, dtype: float64

First 5 rows of target check:
              HDFCBANK  HDFC_ret_next  Target
Date                                         
2015-01-05  500.999146      -0.035745       0
2015-01-06  483.090759      -0.000801       0
2015-01-07  482.703583       0.006618       1
2015-01-08  485.898041      -0.013946       0
2015-01-09  479.121918       0.019295       1


In [11]:
# Step 4: Create features for the neural network
# Why this step comes after Step 3:
# We already created the target column.
# Now we build the input features the model will learn from.
# These features must use only past information so there is no data leakage.

df_nn = ALLData.copy()

# Simple return features
df_nn["ret_1"] = df_nn["HDFCBANK"].pct_change(1)
df_nn["ret_2"] = df_nn["HDFCBANK"].pct_change(2)
df_nn["ret_3"] = df_nn["HDFCBANK"].pct_change(3)
df_nn["ret_5"] = df_nn["HDFCBANK"].pct_change(5)
df_nn["ret_10"] = df_nn["HDFCBANK"].pct_change(10)

# Moving average features
df_nn["ma_5"] = df_nn["HDFCBANK"].rolling(5).mean()
df_nn["ma_20"] = df_nn["HDFCBANK"].rolling(20).mean()
df_nn["ma_gap_5_20"] = df_nn["ma_5"] / df_nn["ma_20"] - 1

# Trend features
df_nn["trend_5"] = df_nn["HDFCBANK"] / df_nn["HDFCBANK"].shift(5) - 1
df_nn["trend_20"] = df_nn["HDFCBANK"] / df_nn["HDFCBANK"].shift(20) - 1

# Volatility features
df_nn["vol_5"] = df_nn["HDFCBANK"].pct_change().rolling(5).std()
df_nn["vol_20"] = df_nn["HDFCBANK"].pct_change().rolling(20).std()
df_nn["vol_ratio"] = df_nn["vol_5"] / df_nn["vol_20"]

# High-low range features
df_nn["high_10"] = df_nn["HDFCBANK"].rolling(10).max()
df_nn["low_10"] = df_nn["HDFCBANK"].rolling(10).min()
df_nn["range_pos_10"] = (df_nn["HDFCBANK"] - df_nn["low_10"]) / (df_nn["high_10"] - df_nn["low_10"])

# Market context features
df_nn["nifty_ret_1"] = df_nn["NIFTY50"].pct_change(1)
df_nn["banknifty_ret_1"] = df_nn["BANKNIFTY"].pct_change(1)
df_nn["vix_chg"] = df_nn["INDIA_VIX"].pct_change(1)
df_nn["usd_chg"] = df_nn["USDINR"].pct_change(1)

# Relative strength features
df_nn["hdfc_minus_nifty"] = df_nn["HDFCBANK"].pct_change(1) - df_nn["NIFTY50"].pct_change(1)
df_nn["hdfc_minus_banknifty"] = df_nn["HDFCBANK"].pct_change(1) - df_nn["BANKNIFTY"].pct_change(1)

# Remove rows with missing values created by rolling and shifting
df_nn = df_nn.dropna()

print("Shape after feature creation:", df_nn.shape)

print("\nColumns now:")
print(df_nn.columns.tolist())

print("\nFirst 5 rows of selected features:")
print(df_nn[[
    "HDFCBANK", "Target", "ret_1", "ret_5", "ret_10",
    "ma_gap_5_20", "vol_5", "vol_20", "range_pos_10",
    "hdfc_minus_nifty", "hdfc_minus_banknifty"
]].head())

Shape after feature creation: (2321, 41)

Columns now:
['HDFCBANK', 'NIFTY50', 'BANKNIFTY', 'SENSEX', 'INDIA_VIX', 'USDINR', 'SP500', 'NASDAQ100', 'DOWJONES', 'NIKKEI225', 'HANGSENG', 'GOLD', 'BRENT_OIL', 'ICICIBANK', 'SBIN', 'AXISBANK', 'KOTAKBANK', 'HDFC_ret_next', 'Target', 'ret_1', 'ret_2', 'ret_3', 'ret_5', 'ret_10', 'ma_5', 'ma_20', 'ma_gap_5_20', 'trend_5', 'trend_20', 'vol_5', 'vol_20', 'vol_ratio', 'high_10', 'low_10', 'range_pos_10', 'nifty_ret_1', 'banknifty_ret_1', 'vix_chg', 'usd_chg', 'hdfc_minus_nifty', 'hdfc_minus_banknifty']

First 5 rows of selected features:
              HDFCBANK  Target     ret_1     ret_5    ret_10  ma_gap_5_20  \
Date                                                                        
2015-02-05  547.803040       0  0.013069 -0.056047  0.034079     0.064921   
2015-02-06  545.140869       0 -0.004860 -0.042425 -0.002833     0.049669   
2015-02-09  535.896301       1 -0.016958 -0.103191 -0.020437     0.021451   
2015-02-10  536.380310       1 

In [13]:
# Step 5: Split data into train, validation, and test
# Why this step comes after Step 4:
# We already created the target and features.
# Now we must separate the data in time order.
# We never shuffle time-series data because future rows must not leak into the past.

feature_cols = [
    "ret_1", "ret_2", "ret_3", "ret_5", "ret_10",
    "ma_5", "ma_20", "ma_gap_5_20",
    "trend_5", "trend_20",
    "vol_5", "vol_20", "vol_ratio",
    "high_10", "low_10", "range_pos_10",
    "nifty_ret_1", "banknifty_ret_1",
    "vix_chg", "usd_chg",
    "hdfc_minus_nifty", "hdfc_minus_banknifty"
]

X = df_nn[feature_cols]
y = df_nn["Target"]

# Time order split:
# First 70% = training
# Next 15% = validation
# Last 15% = testing

train_size = int(len(df_nn) * 0.70)
val_size = int(len(df_nn) * 0.15)

X_train = X.iloc[:train_size]
y_train = y.iloc[:train_size]

X_val = X.iloc[train_size:train_size + val_size]
y_val = y.iloc[train_size:train_size + val_size]

X_test = X.iloc[train_size + val_size:]
y_test = y.iloc[train_size + val_size:]

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

print("\nTrain target balance:")
print(y_train.value_counts(normalize=True).round(4))

print("\nValidation target balance:")
print(y_val.value_counts(normalize=True).round(4))

print("\nTest target balance:")
print(y_test.value_counts(normalize=True).round(4))

Train shape: (1624, 22) (1624,)
Validation shape: (348, 22) (348,)
Test shape: (349, 22) (349,)

Train target balance:
1    0.5148
0    0.4852
Name: Target, dtype: float64

Validation target balance:
1    0.5431
0    0.4569
Name: Target, dtype: float64

Test target balance:
1    0.5043
0    0.4957
Name: Target, dtype: float64


In [15]:
# Step 6: Scale the feature values
# Why this step comes after Step 5:
# We already split the data in time order.
# Now we scale the features so the neural network can learn properly.
# Important: fit the scaler only on the training data to avoid leakage.

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled validation shape:", X_val_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

print("\nFirst 3 scaled train rows:")
print(X_train_scaled[:3])

Scaled train shape: (1624, 22)
Scaled validation shape: (348, 22)
Scaled test shape: (349, 22)

First 3 scaled train rows:
[[ 0.51208369 -1.05197222 -2.01843106 -1.094645    0.39828396 -0.19550048
  -0.4873156   1.3876548  -1.094645    0.84582931  1.58631049  0.83621486
   1.15719664 -0.10466947 -0.20845167 -1.05406947  1.40926268 -0.16299449
  -1.1816388   0.89309864 -0.78708556  0.52657444]
 [-0.2212059   0.2044089  -0.9881935  -0.84008886 -0.10651998 -0.2362025
  -0.46059192  1.05131715 -0.84008886  1.20094997  1.57069748  0.71604382
   1.30875884 -0.10466947 -0.20845167 -1.17700936  0.67574975 -2.22830552
  -0.85021035 -0.51868369 -0.66531997  0.63488265]
 [-0.7160427  -0.6599412  -0.24981978 -1.97565709 -0.34726167 -0.34011904
  -0.43768294  0.42906267 -1.97565709  1.01593624  0.50766471  0.7527665
  -0.09794334 -0.10466947 -0.24977218 -1.3804125   0.276017    0.53228558
  -1.26754157  0.83076721 -0.65770121 -0.85048809]]


In [17]:
# Step 7: Build the first neural network model
# Why this step comes after Step 6:
# We already prepared scaled training, validation, and test data.
# Now we create the neural network structure.
# We keep the model simple first so it is easier to understand and debug.

model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid")
])

# Compile the model
# binary_crossentropy is used for 0/1 classification
# adam is a common optimizer that works well for beginners
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Show model summary
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                1472      
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dropout_1 (Dropout)         (None, 32)                0         
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 3,585
Trainable params: 3,585
Non-trainable params: 0
_________________________________________________________________


In [19]:
# Step 8: Train the neural network
# Why this step comes after Step 7:
# We already built the model.
# Now we fit it on the training data.
# We use early stopping so training stops when validation loss stops improving.

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
51/51 [==============================] - 3s 15ms/step - loss: 0.7166 - accuracy: 0.5006 - val_loss: 0.6923 - val_accuracy: 0.5575
Epoch 2/100
51/51 [==============================] - 0s 6ms/step - loss: 0.7069 - accuracy: 0.5216 - val_loss: 0.7501 - val_accuracy: 0.4684
Epoch 3/100
51/51 [==============================] - 0s 5ms/step - loss: 0.7020 - accuracy: 0.5203 - val_loss: 0.6991 - val_accuracy: 0.4885
Epoch 4/100
51/51 [==============================] - 0s 5ms/step - loss: 0.6886 - accuracy: 0.5357 - val_loss: 0.7028 - val_accuracy: 0.5057
Epoch 5/100
51/51 [==============================] - 0s 7ms/step - loss: 0.6902 - accuracy: 0.5406 - val_loss: 0.7145 - val_accuracy: 0.4770
Epoch 6/100
51/51 [==============================] - 0s 5ms/step - loss: 0.6860 - accuracy: 0.5369 - val_loss: 0.7099 - val_accuracy: 0.4914
Epoch 7/100
51/51 [==============================] - 0s 5ms/step - loss: 0.6776 - accuracy: 0.5671 - val_loss: 0.7042 - val_accuracy: 0.5230
Epoch 8/100


In [21]:
# Step 9: Evaluate the model on the test set
# Why this step comes after Step 8:
# The model has already been trained.
# Now we check how it performs on completely unseen test data.
# We first get probabilities from the model, then convert them into 0/1 labels.

test_prob = model.predict(X_test_scaled)

# Convert probabilities to class labels using 0.5 threshold
test_pred = (test_prob >= 0.5).astype(int).ravel()

print("Test Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Test Precision:", round(precision_score(y_test, test_pred, zero_division=0), 4))
print("Test Recall:", round(recall_score(y_test, test_pred, zero_division=0), 4))
print("Test F1:", round(f1_score(y_test, test_pred, zero_division=0), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

11/11 [==============================] - 0s 4ms/step
Test Accuracy: 0.5043
Test Precision: 0.5043
Test Recall: 1.0
Test F1: 0.6705

Confusion Matrix:
[[  0 173]
 [  0 176]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       173
           1       0.50      1.00      0.67       176

    accuracy                           0.50       349
   macro avg       0.25      0.50      0.34       349
weighted avg       0.25      0.50      0.34       349



C:\Users\Dell\anaconda3\envs\aryanquants\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Dell\anaconda3\envs\aryanquants\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Dell\anaconda3\envs\aryanquants\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [23]:
# Step 10: Check predicted probabilities and try different thresholds
# Why this step comes after Step 9:
# The model is predicting only one class, so we need to see the raw probabilities.
# Then we can test different thresholds and choose a better cutoff.

test_prob = model.predict(X_test_scaled).ravel()

print("Probability summary:")
print("Min:", round(test_prob.min(), 4))
print("Max:", round(test_prob.max(), 4))
print("Mean:", round(test_prob.mean(), 4))

threshold_list = [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]

results = []

for th in threshold_list:
    pred = (test_prob >= th).astype(int)

    results.append({
        "Threshold": th,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    })

threshold_results = pd.DataFrame(results)
print("\nThreshold results:")
print(threshold_results)

11/11 [==============================] - 0s 2ms/step
Probability summary:
Min: 0.5089
Max: 0.732
Mean: 0.603

Threshold results:
   Threshold  Accuracy  Precision    Recall        F1
0       0.30  0.504298   0.504298  1.000000  0.670476
1       0.40  0.504298   0.504298  1.000000  0.670476
2       0.45  0.504298   0.504298  1.000000  0.670476
3       0.50  0.504298   0.504298  1.000000  0.670476
4       0.55  0.507163   0.506494  0.886364  0.644628
5       0.60  0.527221   0.530726  0.539773  0.535211
6       0.70  0.495702   0.500000  0.022727  0.043478


In [25]:
# Step 11: Choose the best threshold using the validation set
# Why this step comes after Step 10:
# We tested several thresholds on the test set and found the model is poorly calibrated.
# Now we should choose the threshold using validation data only.
# This is the correct way because we must not tune based on the test set.

val_prob = model.predict(X_val_scaled).ravel()

print("Validation probability summary:")
print("Min:", round(val_prob.min(), 4))
print("Max:", round(val_prob.max(), 4))
print("Mean:", round(val_prob.mean(), 4))

threshold_list = [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

val_results = []

for th in threshold_list:
    val_pred = (val_prob >= th).astype(int)

    val_results.append({
        "Threshold": th,
        "Accuracy": accuracy_score(y_val, val_pred),
        "Precision": precision_score(y_val, val_pred, zero_division=0),
        "Recall": recall_score(y_val, val_pred, zero_division=0),
        "F1": f1_score(y_val, val_pred, zero_division=0)
    })

val_threshold_results = pd.DataFrame(val_results)
print("\nValidation threshold results:")
print(val_threshold_results)

best_row = val_threshold_results.loc[val_threshold_results["F1"].idxmax()]
best_threshold = best_row["Threshold"]

print("\nBest threshold based on validation F1:", best_threshold)

11/11 [==============================] - 0s 3ms/step
Validation probability summary:
Min: 0.3973
Max: 0.7132
Mean: 0.5719

Validation threshold results:
   Threshold  Accuracy  Precision    Recall        F1
0       0.30  0.543103   0.543103  1.000000  0.703911
1       0.40  0.545977   0.544669  1.000000  0.705224
2       0.45  0.545977   0.544669  1.000000  0.705224
3       0.50  0.557471   0.551622  0.989418  0.708333
4       0.55  0.488506   0.522822  0.666667  0.586047
5       0.60  0.497126   0.581395  0.264550  0.363636
6       0.65  0.465517   0.666667  0.031746  0.060606
7       0.70  0.462644   1.000000  0.010582  0.020942

Best threshold based on validation F1: 0.5


In [27]:
# Step 12: Final test evaluation using the chosen threshold
# Why this step comes after Step 11:
# We used validation data to choose the best threshold.
# Now we apply that threshold once on the unseen test set.
# This is the final honest evaluation of the neural network.

test_prob = model.predict(X_test_scaled).ravel()

test_pred_best = (test_prob >= best_threshold).astype(int)

print("Final test evaluation using threshold:", best_threshold)
print("Test Accuracy:", round(accuracy_score(y_test, test_pred_best), 4))
print("Test Precision:", round(precision_score(y_test, test_pred_best, zero_division=0), 4))
print("Test Recall:", round(recall_score(y_test, test_pred_best, zero_division=0), 4))
print("Test F1:", round(f1_score(y_test, test_pred_best, zero_division=0), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred_best))

print("\nClassification Report:")
print(classification_report(y_test, test_pred_best))

11/11 [==============================] - 0s 8ms/step
Final test evaluation using threshold: 0.5
Test Accuracy: 0.5043
Test Precision: 0.5043
Test Recall: 1.0
Test F1: 0.6705

Confusion Matrix:
[[  0 173]
 [  0 176]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       173
           1       0.50      1.00      0.67       176

    accuracy                           0.50       349
   macro avg       0.25      0.50      0.34       349
weighted avg       0.25      0.50      0.34       349



C:\Users\Dell\anaconda3\envs\aryanquants\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Dell\anaconda3\envs\aryanquants\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Dell\anaconda3\envs\aryanquants\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [29]:
# Step 13: Final conclusion and simple summary
# Why this step comes after Step 12:
# We already trained the model and evaluated it on the test set.
# Now we summarize what happened in simple beginner language.

print("=== NEURAL NETWORK PROJECT SUMMARY ===")
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)
print("\nBest threshold from validation:", best_threshold)

print("\nUse these results for your final notebook conclusion:")
print("1. Check test accuracy")
print("2. Check test precision")
print("3. Check test recall")
print("4. Check test F1")
print("5. Check confusion matrix")
print("6. Compare with naive baseline and earlier models")

print("\nIf performance is near 0.5 accuracy and F1 is not clearly better than baseline,")
print("then the neural network does not show a strong predictive edge for HDFCBANK direction.")

=== NEURAL NETWORK PROJECT SUMMARY ===
Train shape: (1624, 22)
Validation shape: (348, 22)
Test shape: (349, 22)

Best threshold from validation: 0.5

Use these results for your final notebook conclusion:
1. Check test accuracy
2. Check test precision
3. Check test recall
4. Check test F1
5. Check confusion matrix
6. Compare with naive baseline and earlier models

If performance is near 0.5 accuracy and F1 is not clearly better than baseline,
then the neural network does not show a strong predictive edge for HDFCBANK direction.


In [31]:
# Final conclusion
print("Final conclusion:")
print("The neural network was trained on HDFCBANK direction prediction using lagged returns,")
print("trend, volatility, and market context features.")

print("\nThe model trained successfully, but the validation and test results were weak.")
print("Even after threshold checking, the best threshold remained 0.5.")

print("\nThis means the neural network does NOT currently show a strong predictive edge")
print("for next-day HDFCBANK direction.")

print("\nCompared with the naive baseline and earlier models,")
print("the neural network does not clearly outperform chance in a stable way.")

print("\nFinal takeaway:")
print("This is a useful research result, because it shows that the current feature set")
print("and target setup are not enough for reliable trading prediction.")

Final conclusion:
The neural network was trained on HDFCBANK direction prediction using lagged returns,
trend, volatility, and market context features.

The model trained successfully, but the validation and test results were weak.
Even after threshold checking, the best threshold remained 0.5.

This means the neural network does NOT currently show a strong predictive edge
for next-day HDFCBANK direction.

Compared with the naive baseline and earlier models,
the neural network does not clearly outperform chance in a stable way.

Final takeaway:
This is a useful research result, because it shows that the current feature set
and target setup are not enough for reliable trading prediction.


In [33]:
# Tuning order for my neural network project

# 1. Learning rate first
# This is the most important tuning parameter because it controls
# how fast the model learns during training.

# 2. Batch size second
# This decides how many samples the model sees before updating weights.
# Common values are 16, 32, and 64.

# 3. Epochs with early stopping third
# Train for enough epochs, but use early stopping so the model
# stops when validation loss stops improving.

# 4. Hidden layers and neurons
# If the model is underfitting, increase the number of neurons
# or add one more hidden layer.

# 5. Dropout rate
# If the model is overfitting, increase dropout to reduce noise and memorization.

# 6. L2 regularization
# If needed, add L2 penalty to keep weights smaller and improve generalization.

# 7. Architecture change last
# If the dense neural network is still weak, then try a different model
# such as LSTM or GRU.